In [ ]:
import sys
from os.path import join
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
mm = 1.0/2.54/10

repo_root_path = join('..','..')

python_modules_paths = (
    join(repo_root_path, 'src', 'python'),
    'python'
)
for python_modules_path in python_modules_paths:
    if python_modules_path not in sys.path:
        sys.path.append(python_modules_path)

from foam.common import get_wpd_columns
from scm.scm import ilmenite, ilmenite_orig, Abad_SCR, Abad_SCRd, IlmenitePhase, parseReaction, concentration_from_molar_fraction
from labreactor.labreactor import LabReactor
from labreactor.labreactor_analytical import gas_yield_Leion
from labreactor.plotting import plot_labReactor_eff, plot_labReactors_eff, plot_labReactors_exp_eff

plt.style.use(os.path.join(repo_root_path, 'src', 'python', 'clc.mplstyle'))
# TODO: green color (third) is too close to blue (first)
# plt.style.use('tableau-colorblind10')
# ilmenite['act']['H2'].k(T=950+273)
from specie_properties.common import AW, MW

colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
def setup_conversion_plot(xlimr = 0.94, ylimt=1.0):
    fig, ax = plt.subplots(figsize=[90*mm, 85*mm])
    ax.invert_xaxis()
    ax.set(
        xlim = [1,xlimr],
        ylim = [0,ylimt],
        xlabel = '$\omega, -$',
        ylabel = '$\gamma, -$',
    )
    return fig, ax

def chemFullName(chem):
    if chem == 'act':
        return 'activated'
    elif chem == 'pre':
        return 'pre-oxidized'
    else:
        raise ValueError('Unknown chem ' + chem)

# Leion et al. 2008

In [ ]:
def _get_analytical(rhoox, Ro, chem, gas, nVolRate_mLn_min, ms, T, p, xA0):
    phase = IlmenitePhase(rhoox=rhoox, Ro=Ro)
    reaction = ilmenite[chem][gas]
    oxidations = np.linspace(0,1,21)
    Vdot0 = nVolRate_mLn_min * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
    f = np.vectorize(gas_yield_Leion)
    omega = 1 - Ro * (1-oxidations)
    gas_yield = f(phase, reaction, ms, oxidations, T, p, xA0, Vdot0)
    return omega, gas_yield




def plot_analytical(ax, labReactor, *args, **kwargs):
    if labReactor.gas == 'syngas':
        gas = 'CO'
        xA0 = 0.5  # [-] - mole fraction of fuel at the inlet
    else:
        gas = labReactor.gas
        xA0 = 1.0
    p                   = 101325    # [Pa] - pressure
    chem                = labReactor.chemistry
    nVolRate_mLn_min    = labReactor.nVolRate_mLn_min
    T                   = labReactor.T
    Ro                  = labReactor.Ro
    ms                  = labReactor.solidsMass # [kg] - mass of oxygen carrier
    rhoox               = labReactor.rhoOxidized
    # phase = IlmenitePhase(rhoox=rhoox, Ro=Ro)
    # reaction = ilmenite[chem][gas]
    # oxidations = np.linspace(0,1,21)
    # Vdot0 = nVolRate_mLn_min * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
    # f = np.vectorize(gas_yield_Leion)
    # gas_yield = f(phase, reaction, ms, oxidations, T, p, xA0, Vdot0)

    # omega = 1 - Ro * (1-oxidations)
    omega, gas_yield = _get_analytical(rhoox, Ro, chem, gas, nVolRate_mLn_min, ms, T, p, xA0)
    ax.plot(omega, gas_yield, *args, **kwargs)

def plot_analytical_range(ax, labReactor, dT, *args, **kwargs):
    if labReactor.gas == 'syngas':
        gas = 'CO'
        xA0 = 0.5  # [-] - mole fraction of fuel at the inlet
    else:
        gas = labReactor.gas
        xA0 = 1.0
    p                   = 101325    # [Pa] - pressure
    chem                = labReactor.chemistry
    nVolRate_mLn_min    = labReactor.nVolRate_mLn_min
    T                   = labReactor.T
    Ro                  = labReactor.Ro
    ms                  = labReactor.solidsMass # [kg] - mass of oxygen carrier
    rhoox               = labReactor.rhoOxidized

    omega0, gas_yield0 = _get_analytical(rhoox, Ro, chem, gas, nVolRate_mLn_min, ms, T-dT, p, xA0)
    omega1, gas_yield1 = _get_analytical(rhoox, Ro, chem, gas, nVolRate_mLn_min, ms, T+dT, p, xA0)

    ax.fill_between(omega0, gas_yield0, gas_yield1, *args, **kwargs)

## CH4

In [ ]:
fuel = 'CH4'
postfix = '_950C'
casesCH4 = [LabReactor(f'labReactor_{octype}_{fuel}{postfix}') for octype in ['act', 'pre']]

In [ ]:
casesCH4[0].eff_fig_name

In [ ]:
fig, ax = setup_conversion_plot(xlimr = 0.955)

plot_labReactors_exp_eff(ax, casesCH4[0].eff_fig_name, label='Leion et al.')

for i, labReactor in enumerate(casesCH4):
    plot_labReactor_eff(ax, labReactor, color=colors[i], label=chemFullName(labReactor.chemistry))
    plot_analytical(ax, labReactor, color=colors[i], linestyle='--')
    plot_analytical_range(ax, labReactor, 20, color=colors[i], alpha=0.2)

# plot_analytical(ax, LabReactor(f'labReactor_act_CH4_958C'), color=colors[2], linestyle='--')

ax.annotate('increasing\ncycle number', xy=(0.998,0.05), xytext=(0.989, 0.52), arrowprops=dict(arrowstyle='<-'), horizontalalignment='center')

h, l = ax.get_legend_handles_labels()
ax.legend(h[-3:], l[-3:], loc='upper right', framealpha=0.9, frameon=False)

# ax.text(0.09, 1.04, 'solid — sim/exp, dashed — analytical',transform=ax.transAxes, horizontalalignment='left')

ax.set_xticklabels(ax.get_xticklabels()[:-1] + ['1'])
# ax.set_yticklabels(['0'] + ax.get_yticklabels()[1:])

print('Reducing agent conversion as a function of normalized mass of ilmenite from experimental data by Leion et al. [] (dashed lines) and simulation results (solid lines)')
fig.tight_layout()
fig.savefig(f'plots/labReactor_results_Leion2008_CH4.pdf')

## Syngas

In [ ]:
fuel = 'syngas'
postfix = '_950C'
casessyngas = [LabReactor(f'labReactor_{octype}_{fuel}{postfix}') for octype in ['act', 'pre']]

In [ ]:
fig, ax = setup_conversion_plot(xlimr = 0.94)

plot_labReactors_exp_eff(ax, casessyngas[0].eff_fig_name, label='Leion et al.')
# plot_labReactors_eff(ax, casessyngas)
# colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
# for i, labReactor in enumerate(casessyngas):
#     plot_analytical(ax, labReactor, color=colors[i], linestyle='--', label=f'{labReactor.chemistry} analyt.')
for i, labReactor in enumerate(casessyngas):
    plot_labReactor_eff(ax, labReactor, color=colors[i], label=chemFullName(labReactor.chemistry))
    plot_analytical(ax, labReactor, color=colors[i], linestyle='--')
    plot_analytical_range(ax, labReactor, 20, color=colors[i], alpha=0.2)

ax.annotate('', xy=(0.998,0.82), xytext=(0.948, 0.7), arrowprops=dict(arrowstyle='<-'), horizontalalignment='right')
ax.text(0.952, 0.55, 'increasing\ncycle number', horizontalalignment='center')

h, l = ax.get_legend_handles_labels()
ax.legend(h[-3:], l[-3:], loc='lower left', framealpha=0.9, frameon=False)

# ax.text(0.09, 1.04, 'solid — sim/exp, dashed — analytical',transform=ax.transAxes, horizontalalignment='left')

ax.set_xticklabels(ax.get_xticklabels()[:-1] + ['1'])
# ax.set_yticklabels(['0'] + ax.get_yticklabels()[1:])

print('Reducing agent conversion as a function of normalized mass of ilmenite from experimental data by Leion et al. [] (dashed lines) and simulation results (solid lines)')
fig.tight_layout()
fig.savefig(f'plots/labReactor_results_Leion2008_syngas.pdf')

# Berguerand 2011

In [ ]:
def get_analytical(fuel, T, mass):
    if fuel == 'syngas':
        fuel = 'CO'
        xA0 = 0.5  # [-] - mole fraction of fuel at the inlet
    else:
        fuel = fuel
        xA0 = 1.0
    p                   = 101325    # [Pa] - pressure
    chem                = 'act'
    nVolRate_mLn_min    = 900
    T                   = T
    Ro                  = 0.033
    ms                  = mass # [kg] - mass of oxygen carrier
    phase = IlmenitePhase(rhoox=3000, Ro=Ro)
    reaction = ilmenite[chem][fuel]
    oxidations = np.linspace(0,1,121)
    Vdot0 = nVolRate_mLn_min * 1e-6 / 60 * T / ( 25 + 273.15 )   # [m^3/s] - total volumetric gas flow at inlet
    f = np.vectorize(gas_yield_Leion)
    gas_yield = f(phase, reaction, ms, oxidations, T, p, xA0, Vdot0)
    omega = 1 - Ro * (1-oxidations)
    return omega, gas_yield


def plot_analytical_3g(ax, fuel, T, *args, **kwargs):
    omega, gas_yield = get_analytical(fuel, T, 0.003)
    h, = ax.plot(omega, gas_yield, *args, **kwargs)
    return h

def correct_pass_time(omega, gamma, pass_time=0):
    gamma = gamma.loc[pass_time:].to_numpy()
    omega = omega.iloc[:len(gamma)].to_numpy()
    return omega, gamma

dfs_validation = {
    'H2': pd.read_csv(join('validation',f'Berguerand2011_fig5.csv'), skiprows=2, names=get_wpd_columns(join('validation',f'Berguerand2011_fig5.csv'))),
    'CO': pd.read_csv(join('validation',f'Berguerand2011_fig6.csv'), skiprows=2, names=get_wpd_columns(join('validation',f'Berguerand2011_fig6.csv'))),
}

## H2 3g gas yield vs conversion

In [ ]:
TsdegC = [800,900,1050]
cdH2 = {T: LabReactor(f'labReactor_act_H2_3g_{T}C') for T in TsdegC}

In [ ]:
fig, ax = setup_conversion_plot(xlimr = 0.965)

omega = np.linspace(0.967, 1.0, 331)
handles_sim = []
handles_model = []
colors_table = [colors[2], colors[1], colors[0]]  # 800, 900, 1050

for idx, (TdegC, c) in enumerate(zip(TsdegC, [cdH2[800], cdH2[900], cdH2[1050]])):
    print(TdegC)
    omega_corr, gamma = correct_pass_time(c.omega(), c.gamma(), pass_time=1.3)
    gamma = np.interp(omega, np.flip(omega_corr), np.flip(gamma))
    h_sim, = ax.plot(omega, gamma, linestyle='--', color=colors_table[idx])
    handles_sim.append(h_sim)
    h_model = ax.plot(*get_analytical('H2', TdegC + 273.15, 0.003), linestyle=':', color=colors_table[idx])[0]
    handles_model.append(h_model)

hexp800, = ax.plot(dfs_validation['H2']['800C_X'], dfs_validation['H2']['800C_Y'], color=colors[2], linestyle='-', label='800 °C exp.')
hexp900, = ax.plot(dfs_validation['H2']['900-1050C_X'], dfs_validation['H2']['900-1050C_Y'], color=colors[1], linestyle='-', label='900-1050 °C exp.')
hexp1050 = None  # No separate exp. for 1050

from matplotlib.patches import Rectangle
extra = Rectangle((0, 0), 1, 1, fc="w", fill=False, edgecolor='none', linewidth=0)

# Table: 1st row is headers, then each temp row: exp, sim, model
legend_handles = [
    extra, extra, extra, extra,
    extra, hexp800, handles_sim[0], handles_model[0],
    extra, hexp900, handles_sim[1], handles_model[1],
    extra, hexp900, handles_sim[2], handles_model[2]
]
legend_labels = [
    "$T_\mathrm{inlet}$, °C", "Experiment", "Simulation", "Analytical",
    "800", "", "", "",
    "900", "", "", "",
    "1050", "", "", ""
]

ax.legend(legend_handles, legend_labels, loc='lower left', ncol=4, handletextpad=-2, columnspacing=1.2, frameon=True, fontsize=8)

fig.tight_layout()
fig.savefig(f'plots/labReactor_results_Leion2010_3g_H2.pdf')
print('Reducing agent conversion as a function of normalized mass of ilmenite from experimental data by Leion et al. [] and simulation results over selected largest and smallest inlet temperatures.')

## CO 3g gas yield vs conversion

In [ ]:
TsdegC = [800,900,950,1000,1050]
cd = {T: LabReactor(f'labReactor_act_CO_3g_{T}C') for T in TsdegC}

In [ ]:
from matplotlib.patches import Rectangle

fig, ax = setup_conversion_plot(xlimr = 0.965, ylimt=1.11)

omega = np.linspace(0.967, 1.0, 331)

for TdegC, c in reversed(cd.items()):
    print(TdegC)
    omegaorig, gamma = correct_pass_time(c.omega(),  c.gamma(),  pass_time=1.3)
    gamma = np.interp(omega, np.flip(omegaorig),  np.flip(gamma))
    
    h = ax.plot(omega, gamma, linestyle='--')
    plot_analytical(ax, c, color=h[0]._color, linestyle=':')

    ax.plot(dfs_validation['CO'][f'{TdegC}C_X'], dfs_validation['CO'][f'{TdegC}C_Y'], linestyle='-', label=r'$'+f'{TdegC}'+r'\degree\mathrm{C}$', color=h[0]._color)

extra = Rectangle((0, 0), 1, 1, fc="w", fill=False, edgecolor='none', linewidth=0)

legend_handles = [
    extra, extra, extra, extra,
    extra,  # 800 °C row label
    ax.lines[14], ax.lines[12], ax.lines[13],  # 800 °C: exp, sim, model
    extra,  # 900 °C row label
    ax.lines[11], ax.lines[9], ax.lines[10],  # 900 °C: exp, sim, model
    extra,  # 950 °C row label
    ax.lines[8], ax.lines[6], ax.lines[7],  # 950 °C: exp, sim, model
    extra,  # 1000 °C row label
    ax.lines[5], ax.lines[3], ax.lines[4],  # 1000 °C: exp, sim, model
    extra,  # 1050 °C row label
    ax.lines[2], ax.lines[0], ax.lines[1],  # 1050 °C: exp, sim, model
]
legend_labels = [
    "$T_\mathrm{inlet}$, °C", "Experiment", "Simulation", "Analytical",
    "800", "", "", "",
    "900", "", "", "",
    "950", "", "", "",
    "1000", "", "", "",
    "1050", "", "", "",
]

ax.legend(legend_handles, legend_labels, loc='upper left', ncol=6, handletextpad=-2, columnspacing=1.2, frameon=True, fontsize=8)
# ax.text(-0.09, 1.08, 'solid — exp., dashed — sim., dotted — analytical',transform=ax.transAxes, horizontalalignment='left')
fig.tight_layout()

fig.savefig('plots/labReactor_results_Leion2010_3g_CO.pdf')